In [204]:
"""
This script creates artificial data for a discrete choice problem.
Assume there are three modes of transportation to choose from. Six fixed
variables were designed as significant and five as non-significant.
Seven random variables (five normal, one uniform, one triangular) were
designed as significant.
Three normal variables were correlated.
Two normal variables were non-linearly transformed.
"""


'\nThis script creates artificial data for a discrete choice problem.\nAssume there are three modes of transportation to choose from. Six fixed\nvariables were designed as significant and five as non-significant.\nSeven random variables (five normal, one uniform, one triangular) were\ndesigned as significant.\nThree normal variables were correlated.\nTwo normal variables were non-linearly transformed.\n'

In [205]:

import numpy as np

import random





import statsmodels.api as sm
from scipy.stats import multivariate_normal

In [206]:
import numpy as np
import pandas as pd
import scipy.stats as ss

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
is_correlated = 0

is_hpc_test = 1

    
if is_correlated:
    name_2 = 'artificial_mixed_corr_2023_MOOF.csv'
    seed = 122
else:
    name_2 = 'artificial_ZA.csv'
    seed = 122
    
is_panel = 0
if is_panel:
    name_2 = 'panel_synth.csv'
    seed = 122
    
if is_hpc_test ==1:
    name_2 = 'var_test.csv'
    seed = 1230


In [207]:
def noise(n_obs, perc=1, random_state=None):
    random_state = random_state or np.random
    noise_vec = random_state.normal(0, 1, n_obs)
    return noise_vec

In [208]:
def random_col(N, P, J, random_state=None, noise_num = 0.5):
    rand_nums = random_state.randint(low=0, high=2, size=(N,))
    return np.tile(rand_nums, P) + noise_num*noise(N*P*J, random_state=random_state)

def generate_random_df(N, P, J, num_fixed=0, num_isvars=0, num_randvars=0, random_state=np.random, add_constant = 1, non_sig_num = None, G = None):
    df = pd.DataFrame()
    df['id'] = np.repeat(np.arange(1, (N*P+1)), J)
    df['ind_id'] = np.repeat(np.arange(1, N+1), P*J)
    df['alt'] = np.tile(np.arange(1, J+1), N*P)
    

    varnames = []
    for i in range(num_isvars):
        coef_name = 'added_isvar' + str(i+1)
        varnames.append(coef_name)
        col_vals = np.repeat(random_state.random(N*P)*100, J)
        for j in range(J):
            if j == 0:
                df[coef_name] = col_vals
            else:
                df[coef_name + "." + str(j+1)] = col_vals
    
    
    for i in range(num_fixed):
        coef_name = 'added_fixed' + str(i+1)
        varnames.append(coef_name)
        df[coef_name] = random_col(N, P, J, random_state=random_state, noise_num=0)
    
    

    for i in range(num_randvars):
        coef_name = 'added_random' + str(i+1)
        varnames.append(coef_name)
        df[coef_name] = random_col(N, P, J, random_state=random_state, noise_num=0)
    if add_constant:
        df['constant'] = np.repeat(1, N*P)
    
    if G is not None:
        df['group'] = np.repeat(np.arange(1, N/G+1),P*J*G)
    
    return df, varnames


In [209]:

N =1000  # Number of observations
P = 5  # Number of choices per individual
J = 1  # Number of alternatives
G =None
num_fixed = 3
num_isvars = 100
num_nonsig = 3
num_randvars = 3

random_state = np.random.RandomState(seed)
np.random.seed(seed)

random.seed(seed)
df, varnames = generate_random_df(N, P, J, num_fixed=num_fixed, num_isvars=num_isvars,
                                  num_randvars=num_randvars, random_state=random_state, non_sig_num = num_nonsig, G = G)

# Shift values <-2 for as inverse boxcox transform will be applied





C:\Users\n9471103\AppData\Local\Temp\ipykernel_9596\2243704842.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[coef_name] = col_vals
C:\Users\n9471103\AppData\Local\Temp\ipykernel_9596\2243704842.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[coef_name] = col_vals
C:\Users\n9471103\AppData\Local\Temp\ipykernel_9596\2243704842.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at onc

In [210]:
df.head(40)

,id,ind_id,alt,added_isvar1,added_isvar2,added_isvar3,added_isvar4,added_isvar5,added_isvar6,added_isvar7,...,added_isvar98,added_isvar99,added_isvar100,added_fixed1,added_fixed2,added_fixed3,added_random1,added_random2,added_random3,constant
0,1,1,1,27.702631,8.250332,1.335045,19.668643,69.399588,52.250805,37.665087,...,77.559374,18.195134,89.223516,0.0,0.0,1.0,1.0,0.0,0.0,1
1,2,1,1,36.855193,63.488482,2.867782,14.478364,24.989083,15.416925,26.321013,...,42.016216,74.863260,96.964760,1.0,1.0,0.0,1.0,1.0,1.0,1
2,3,1,1,64.431478,94.294978,2.722058,78.905011,80.649818,46.307223,81.210143,...,65.290849,88.406905,15.525862,0.0,1.0,0.0,0.0,0.0,0.0,1
3,4,1,1,78.019793,62.627321,30.164255,20.228891,93.375935,91.292542,0.528169,...,51.210977,98.395701,66.050734,1.0,1.0,1.0,0.0,1.0,1.0,1
4,5,1,1,50.860458,16.659229,14.830344,16.756275,85.428285,32.846570,87.290527,...,3.598690,47.173403,8.470648,1.0,1.0,0.0,0.0,1.0,0.0,1
5,6,2,1,52.375554,70.127974,96.337231,70.578086,94.008047,3.926842,89.611070,...,53.685998,59.635124,45.031609,0.0,1.0,0.0,1.0,0.0,0.0,1
6,7,2,1,84.079088,52.447313,35.768137,1.779100,69.643145,99.306193,6.847612,...,4.604300,94.215302,65.849060,1.0,0.0,1.0,1.0,0.0,0.0,1
7,8,2,1,36.703687,8.634309,18.924871,31.014007,15.204269,21.326905,91.205337,...,53.257650,49.941925,54.086433,0.0,1.0,0.0,1.0,1.0,0.0,1
8,9,2,1,67.039217,99.343544,26.575625,56.294758,81.629336,59.901067,24.452185,...,76.386541,25.358311,29.791318,1.0,1.0,1.0,1.0,0.0,1.0,1
9,10,2,1,83.824478,84.914658,31.570155,85.930824,60.465794,14.833367,47.692898,...,83.788762,74.650095,93.744949,0.0,0.0,1.0,1.0,0.0,0.0,1


In [211]:
# Define coefficients (betas)
# Fixed betas
fixed_coefs = [ random_state.choice([-1,1, -1]) *random_state.uniform(1, 4) for i in range(num_fixed)]
fixed_coefs = np.array(fixed_coefs)



fixed_coefs = list(fixed_coefs)


In [212]:
print(fixed_coefs)


print(df.tail)

[-2.159522485603989, 2.522309999445296, 1.5342410014505825]
<bound method NDFrame.tail of         id  ind_id  alt  added_isvar1  added_isvar2  added_isvar3  \
0        1       1    1     27.702631      8.250332      1.335045   
1        2       1    1     36.855193     63.488482      2.867782   
2        3       1    1     64.431478     94.294978      2.722058   
3        4       1    1     78.019793     62.627321     30.164255   
4        5       1    1     50.860458     16.659229     14.830344   
...    ...     ...  ...           ...           ...           ...   
4995  4996    1000    1     96.833956     20.003984     31.387005   
4996  4997    1000    1     55.053912     99.104090     39.985465   
4997  4998    1000    1     29.831811     53.727896      2.512612   
4998  4999    1000    1     73.209083     66.311940     77.400841   
4999  5000    1000    1     39.853780     29.344621     73.478738   

      added_isvar4  added_isvar5  added_isvar6  added_isvar7  ...  \
0        19.

In [213]:
# Random mean between -1.5 and 1.5, excluding -.1 - .1 as hard to detect effect
random_coefs_mean = [random_state.choice([-1,1]) * random_state.uniform(.5, 1.5) for i in range(num_randvars)]
random_coefs_sd = [random_state.uniform(1.0, 1.5) for i in range(num_randvars)]
print(random_coefs_sd)
cov_mat = np.diag(random_coefs_sd)
cov_mat[0, 1] = cov_mat[1, 0] = 0.25
cov_mat[0, 2] = cov_mat[2, 0] = 0.4
cov_mat[1, 2] = cov_mat[2, 1] = 0.5

random_coefs_uniform_a = 0
random_coefs_uniform_b = random_state.uniform(1, 2)

random_coefs_tri_left = 0
random_coefs_tri_right = random_state.uniform(1, 2)
random_coefs_tri_mode = random_coefs_tri_right/2

rand_coefs = [np.array([]) for i in range(num_randvars)]
if G is None:
    G = 1    
a =int( N/G)
    
for i in range(a):
    res_normal = random_state.multivariate_normal(random_coefs_mean, cov_mat)
    res_uniform = np.array([random_state.uniform(random_coefs_uniform_a, random_coefs_uniform_b)])
    res_triangular = np.array([random_state.triangular(random_coefs_tri_left, random_coefs_tri_mode, random_coefs_tri_right)])
    res = np.concatenate((res_normal, res_uniform, res_triangular))

    for r in range(num_randvars):
       # print(res[r])
        rand_coefs[r] = np.append(rand_coefs[r], np.repeat(res[r], P*J*G))
        #print(rand_coefs[r])


[1.3765698421369872, 1.0553214740177115, 1.092727730664783]


In [214]:

print(random_coefs_mean)

print(random_coefs_sd)
print(cov_mat)
print(res)

[1.2892322111608587, -0.8555784877119469, -0.9523803922370649]
[1.3765698421369872, 1.0553214740177115, 1.092727730664783]
[[1.37656984 0.25       0.4       ]
 [0.25       1.05532147 0.5       ]
 [0.4        0.5        1.09272773]]
[ 2.41802165 -0.70611528 -2.36461465  0.25757292  0.48540773]


In [215]:
random_coefs_uniform_a, random_coefs_uniform_b

(0, 1.0826220687830657)

In [216]:
random_coefs_tri_left, random_coefs_tri_mode, random_coefs_tri_right

(0, 0.7069714117959995, 1.413942823591999)

In [217]:
B_fixed = [np.repeat(f_coef, P*N*J) for f_coef in fixed_coefs]
B_const = [np.repeat(-2, P*N*J)]
if is_correlated:
    B_const = [np.repeat(2, P*N*J)]
else:
    B_const = [np.repeat(-5, P*N*J)]
    


# Convert betas to matrix for easy product
B = [B_fixed, rand_coefs, B_const]
B = [B_i for B_i in B if B_i != []]
B = np.vstack(B).T

In [218]:
# Visualise values after non-linear transformation
# import matplotlib.pyplot as plt
B.shape



(5000, 7)

In [219]:
# Multiply and generate probability
isvars = ['added_isvar' + str(i+1) for i in range(num_isvars)]

X = df.values[:, 3+num_isvars:num_fixed+num_randvars+2+num_isvars+2]  # Extract only necessary columns
#print(X.shape)
#print(X[30, -1])
print(X[1, :])
XB = (X*B).sum(axis=1).reshape(N*P, J)
scale = 1
dispersion = 3

eps = np.random.gumbel(0, 1, (N*P, J))
print(max(XB))
#XB = np.clip(XB, None, 7)
eXB = np.exp(XB).ravel()
dispersion = 10
scale = 1

xg = np.random.gamma(dispersion, scale, N*P)
print('is ', np.mean(xg))
#xg = np.array(rgamma(N, dispersion, dispersion))
xbg = eXB
#xbg = eXB

print(max(eXB))
nbyo = np.random.poisson(xbg)

# Use monte carlo simulation to predict choice
# y = np.apply_along_axis(lambda p: np.eye(J, dtype='int64')[np.argmax(p)], 1, prob).reshape(N*P*J,)
# y = y.reshape(N*P*J,)
print('max is', max(nbyo))
y = []
U = XB + eps

for row in U:
    idx = np.argmax(row)
    row_i = np.zeros(J)
    row_i[idx] = 1
    y.append(row_i)
y = np.array(y)
y = np.squeeze(y.reshape(-1, 1))

df['Y'] = nbyo
count = len(list(filter(lambda x: x != 0, nbyo)))
print(count)  # Output: 4


# Apply non-linear transformations for boxcox testing
save = "C:/Users/n9471103/source/repos/MetaCount/" +name_2
# Save to CSV
print(df.head())
df = df.drop(columns = ['id'])
print(df.head())
df.to_csv(save, index=False)

[1. 1. 0. 1. 1. 1. 1.]
[6.37399054]
is  10.024064815142806
586.3931937676402
max is 556
507
   id  ind_id  alt  added_isvar1  added_isvar2  added_isvar3  added_isvar4  \
0   1       1    1     27.702631      8.250332      1.335045     19.668643   
1   2       1    1     36.855193     63.488482      2.867782     14.478364   
2   3       1    1     64.431478     94.294978      2.722058     78.905011   
3   4       1    1     78.019793     62.627321     30.164255     20.228891   
4   5       1    1     50.860458     16.659229     14.830344     16.756275   

   added_isvar5  added_isvar6  added_isvar7  ...  added_isvar99  \
0     69.399588     52.250805     37.665087  ...      18.195134   
1     24.989083     15.416925     26.321013  ...      74.863260   
2     80.649818     46.307223     81.210143  ...      88.406905   
3     93.375935     91.292542      0.528169  ...      98.395701   
4     85.428285     32.846570     87.290527  ...      47.173403   

   added_isvar100  added_fixed1  add

C:\Users\n9471103\AppData\Local\Temp\ipykernel_9596\223084392.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Y'] = nbyo


In [220]:
np.array([0])

array([0])

In [221]:
import numpy as np
from scipy.integrate import quad
from scipy.stats import lognorm

def poisson_lognormal_pmf(x, mu, sigma):
    def integrand(lam):
        return np.exp(-lam) * (lam**x) / np.math.factorial(x) * lognorm.pdf(lam, sigma, scale=np.exp(mu))
    return quad(integrand, 0, np.inf)[0]

# Example usage
x = 1
mu = 1
sigma = 1
pmf = poisson_lognormal_pmf(x, mu, sigma)
print("PMF for x = {}: {:.6f}".format(x, pmf))


PMF for x = 1: 0.175733


In [222]:
# np.linalg.norm(model.coeff_[:11] - fixed_coefs)